# Lesson 7.5 - Capstone: End-to-End Agentic AI System (template)

## Capstone Brief

Use this notebook as a reusable skeleton for portfolio projects.

### Checklist

- Problem statement with baseline metrics
- Architecture sketch and component ownership
- Evaluation plan (quality + cost + reliability)
- Risk controls and rollback strategy


## Project Template: Scope Definition

Fill these fields before implementation:

- **Domain**: (support, operations, legal, healthcare, etc.)
- **Primary user**:
- **Workflow to automate**:
- **High-risk actions**:
- **Human approval points**:
- **Success KPIs**:


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, List
import logging
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("capstone")


## Tiny RAG Pipeline Skeleton


In [ ]:
CORPUS = [
    "Refunds above $5,000 require finance approval.",
    "Enterprise support SLA is 2 hours for severity-1 incidents.",
    "Customer onboarding checklist includes KYC, contract, and provisioning.",
]

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(CORPUS)


def retrieve_context(query: str, top_k: int = 2) -> List[str]:
    q = vectorizer.transform([query])
    sims = cosine_similarity(q, X).flatten()
    idx = np.argsort(-sims)[:top_k]
    return [CORPUS[i] for i in idx]


## Agent Loop Skeleton


In [ ]:
@dataclass
class CapstoneState:
    goal: str
    retrieved: List[str] = field(default_factory=list)
    actions: List[str] = field(default_factory=list)
    output: str = ""


def plan(state: CapstoneState) -> str:
    if not state.retrieved:
        return "retrieve"
    if not state.output:
        return "synthesize"
    return "finish"


def act(state: CapstoneState, action: str) -> None:
    if action == "retrieve":
        state.retrieved = retrieve_context(state.goal)
        state.actions.append("retrieval_done")
    elif action == "synthesize":
        joined = " ".join(state.retrieved)
        state.output = f"Draft answer based on evidence: {joined}"
        state.actions.append("synthesis_done")
    elif action == "finish":
        state.actions.append("completed")


def run_capstone(goal: str, max_steps: int = 5) -> CapstoneState:
    state = CapstoneState(goal=goal)
    for _ in range(max_steps):
        action = plan(state)
        act(state, action)
        logger.info("action=%s actions_so_far=%s", action, state.actions)
        if action == "finish":
            break
    return state


state = run_capstone("How should we handle a high-value refund for enterprise customer?")
state


## Monitoring Hooks Template

Production systems should log:

- request id
- retrieval hits/misses
- model latency
- tool errors
- policy violations

Add tracing IDs so each user request can be replayed during incident analysis.


## Business Case Studies & Exceptions

### Case Study A: Enterprise Document Assistant

A capstone team built a retrieval-grounded assistant over policy docs and added citation checks. The project stood out because it included quality metrics and failure analysis, not only a UI demo.

### Case Study B: Agentic Support Orchestrator

Another team automated triage and draft responses but kept finance/legal decisions human-approved. They measured resolution-time reduction and escalation precision.

### Exceptions

- Some clients require on-prem model hosting and strict access controls.
- If source data quality is weak, capstone scope should prioritize data cleanup first.


## Interview Questions & Answers

1. **What was the problem you solved in your capstone?**  
   A high-friction workflow where retrieval and agent orchestration improved speed and consistency.
2. **How did you design architecture?**  
   API -> orchestrator -> retrieval + agent loop -> guardrails -> logging.
3. **How did you evaluate success?**  
   Task accuracy, latency, cost per task, and user acceptance.
4. **How did you handle failures?**  
   Retries, fallbacks, and human review for high-risk outputs.
5. **Why include monitoring early?**  
   It enables debugging and trust in demo-to-production transition.
6. **How did you reduce hallucinations?**  
   Retrieval grounding and evidence-based synthesis.
7. **What was your trade-off decision?**  
   Simpler architecture first, then incremental complexity where metrics justified it.
8. **How did you control runtime cost?**  
   Limited context size and bounded action loops.
9. **What production feature would you add next?**  
   Automated evaluation harness with regression tests.
10. **How did you explain ROI?**  
   Compared baseline process effort with post-system efficiency and risk reduction.
